# 04. 머신러닝 에이전트 — sklearn 기반 모델 학습 자동화

## 학습 목표

- sklearn 기반 커스텀 ML 도구 3개(데이터 로드, 학습, 평가)를 정의한다
- 서브에이전트(data-preprocessor)로 전처리 작업을 위임한다
- 에이전트가 여러 모델을 비교하고 최적 모델을 추천하도록 한다
- v1 미들웨어(ToolRetryMiddleware, ModelCallLimitMiddleware)를 적용한다


## 개요

| 항목 | 내용 |
|------|------|
| **프레임워크** | Deep Agents + scikit-learn |
| **커스텀 도구** | `data_loader`, `train_model`, `evaluate_model` |
| **서브에이전트** | `data-preprocessor` (전처리 전문) |
| **백엔드** | `LocalShellBackend(root_dir=".", virtual_mode=True)` |
| **스킬** | `skills/ml-pipeline/SKILL.md` — ML 파이프라인 워크플로 + 보고 형식 |

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요"


In [2]:
# Observability 설정 (선택)
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## 1단계: sklearn 데이터셋 소개

scikit-learn은 학습용 내장 데이터셋을 제공합니다. 이 노트북에서는 두 가지를 사용합니다:

| 데이터셋 | 샘플 수 | 특성 수 | 클래스 수 | 설명 |
|---------|---------|---------|----------|------|
| `iris` | 150 | 4 | 3 | 붓꽃 품종 분류 |
| `wine` | 178 | 13 | 3 | 와인 품종 분류 |


## 2단계: data_loader 도구

데이터셋 이름을 받아 sklearn 내장 데이터를 로드하고, 기본 정보를 반환합니다.


In [4]:
from langchain.tools import tool
from sklearn.datasets import load_iris, load_wine

@tool
def data_loader(dataset_name: str) -> str:
    """sklearn 내장 데이터셋을 로드합니다. 'iris' 또는 'wine'."""
    loaders = {"iris": load_iris, "wine": load_wine}
    if dataset_name not in loaders:
        return f"지원 데이터셋: {list(loaders.keys())}"
    data = loaders[dataset_name]()
    return f"데이터셋: {dataset_name}\n샘플: {data.data.shape[0]}\n특성: {data.feature_names}\n클래스: {list(data.target_names)}"


## 3단계: train_model 도구

데이터셋과 알고리즘을 받아 모델을 학습하고 테스트 정확도를 반환합니다. `train_test_split`으로 80:20 분할합니다.


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

@tool
def train_model(dataset_name: str, algorithm: str) -> str:
    """모델을 학습합니다. algorithm: 'random_forest', 'svm', 'logistic'."""
    loaders = {"iris": load_iris, "wine": load_wine}
    models = {"random_forest": RandomForestClassifier(n_estimators=100, random_state=42), "svm": SVC(random_state=42), "logistic": LogisticRegression(max_iter=1000, random_state=42)}
    if dataset_name not in loaders or algorithm not in models:
        return f"지원: datasets={list(loaders.keys())}, algorithms={list(models.keys())}"
    data = loaders[dataset_name]()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
    clf = models[algorithm].fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    return f"{algorithm} on {dataset_name}: accuracy={acc:.4f}"


## 4단계: evaluate_model 도구

5-fold 교차 검증으로 모델의 일반화 성능을 평가합니다. 단일 테스트셋보다 더 신뢰할 수 있는 성능 추정을 제공합니다.


In [6]:
from sklearn.model_selection import cross_val_score
import numpy as np

@tool
def evaluate_model(dataset_name: str, algorithm: str) -> str:
    """5-fold 교차 검증으로 모델을 평가합니다."""
    loaders = {"iris": load_iris, "wine": load_wine}
    models = {"random_forest": RandomForestClassifier(n_estimators=100, random_state=42), "svm": SVC(random_state=42), "logistic": LogisticRegression(max_iter=1000, random_state=42)}
    if dataset_name not in loaders or algorithm not in models:
        return "지원하지 않는 데이터셋 또는 알고리즘입니다."
    data = loaders[dataset_name]()
    scores = cross_val_score(models[algorithm], data.data, data.target, cv=5)
    return f"{algorithm} on {dataset_name}: mean_cv={scores.mean():.4f}, std={scores.std():.4f}"


## 5단계: 서브에이전트 — data-preprocessor

서브에이전트는 메인 에이전트가 위임하는 전문 작업자입니다. `data-preprocessor`는 데이터 전처리(스케일링, 결측치 처리 등)를 담당합니다.

서브에이전트의 장점:
- 메인 에이전트의 컨텍스트를 깨끗하게 유지
- 전문 도메인에 맞는 시스템 프롬프트 적용
- 결과만 메인 에이전트에 전달


In [7]:
preprocessor_subagent = {
    "name": "data-preprocessor",
    "description": "데이터셋을 로드하고 기본 정보를 분석합니다",
    "system_prompt": "당신은 데이터 분석 전문가입니다.\n데이터셋을 로드하고 특성을 파악하세요.\n결과는 간결하게 요약하세요.",
    "tools": [data_loader],
}

## 6단계: ML 에이전트 생성 (v1 미들웨어)

3개의 ML 도구와 서브에이전트를 조합하여 에이전트를 생성합니다. v1 미들웨어로 안정성을 강화합니다:

| 미들웨어 | 역할 |
|---------|------|
| `ToolRetryMiddleware` | sklearn 도구 실패 시 자동 재시도 (최대 2회) |
| `ModelCallLimitMiddleware` | 무한 루프 방지 — 최대 15회 모델 호출 제한 |


In [8]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    ModelCallLimitMiddleware,
)
from prompts import ML_AGENT_PROMPT

ml_agent = create_deep_agent(
    model=model,
    tools=[data_loader, train_model, evaluate_model],
    subagents=[preprocessor_subagent],
    system_prompt=ML_AGENT_PROMPT,
    backend=LocalShellBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=15),
    ],
)

Prompt 'rag-agent-label:production' not found during refresh, evicting from cache.


Prompt 'sql-agent-label:production' not found during refresh, evicting from cache.


Prompt 'data-analysis-agent-label:production' not found during refresh, evicting from cache.


Prompt 'ml-agent-label:production' not found during refresh, evicting from cache.


Prompt 'deep-research-agent-label:production' not found during refresh, evicting from cache.


## 7단계: 모델 비교 요청

에이전트에게 여러 알고리즘을 비교해달라고 요청합니다. 에이전트는 `train_model`과 `evaluate_model`을 반복 호출하여 결과를 비교합니다.


In [9]:
response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "iris 데이터셋에서 random_forest, svm, logistic 세 알고리즘을 비교해주세요. 교차 검증 결과로 최적 모델을 추천해주세요."}]},
    config=lf_config,
)
print(response["messages"][-1].content)


교차 검증 결과는 다음과 같습니다:

- Random Forest: 평균=0.9667, 표준편차=0.0211
- SVM: 평균=0.9667, 표준편차=0.0211
- Logistic Regression: 평균=0.9733, 표준편차=0.0249

이 결과를 보면, Logistic Regression 모델이 가장 높은 평균 정확도를 기록했습니다. 표준편차도 세 모델 모두 유사 범위 내에 있어 성능의 일관성 역시 좋습니다.

최적 모델 추천: Logistic Regression  
이유: 세 모델 중 평균 정확도가 가장 높으며, 표준편차도 크게 차이나지 않아 안정적으로 좋은 성능을 보입니다.


## 8단계: execute로 커스텀 분석 코드 실행

`LocalShellBackend`는 `execute` 빌트인 도구를 제공합니다. 에이전트에게 커스텀 도구에 없는 분석을 요청하면, `execute`로 직접 Python 코드를 작성하여 실행합니다.

> 이것이 현업에서의 핵심 패턴입니다: **커스텀 도구 = 자주 쓰는 표준 작업**, **execute = 자유로운 탐색적 분석**

In [10]:
chunks = []
for ns, chunk in ml_agent.stream(
    {"messages": [{"role": "user", "content": "execute로 Python 코드를 실행하여 iris 데이터셋의 특성 간 상관관계를 분석해주세요. numpy와 sklearn을 사용하세요."}]},
    config=lf_config,
    subgraphs=True,
):
    chunks.append((ns, chunk))

# 스트리밍 결과 요약 출력
print(f"총 {len(chunks)}개 청크 수신", flush=True)
last = chunks[-1][1] if chunks else {}
if hasattr(last, "get") and last.get("messages"):
    print(last["messages"][-1].content, flush=True)

총 31개 청크 수신


## 도구 직접 테스트

에이전트 없이 도구를 직접 호출하여 결과를 확인할 수 있습니다. 디버깅이나 도구 검증에 유용합니다.


## 요약

| 항목 | 핵심 |
|------|------|
| **ML 도구** | `data_loader` (로드) → `train_model` (학습) → `evaluate_model` (평가) |
| **알고리즘** | RandomForest, SVM, LogisticRegression |
| **서브에이전트** | `data-preprocessor` — 전처리 전문가 위임 |
| **평가** | 5-fold `cross_val_score` — 일반화 성능 추정 |

---

**참고 문서:**
- `docs/deepagents/07-subagents.md`
- `docs/deepagents/06-backends.md`
- [scikit-learn 공식 문서](https://scikit-learn.org/stable/)

**다음 단계:** → [05_deep_research_agent.ipynb](./05_deep_research_agent.ipynb): 딥 리서치 에이전트를 구축합니다.
